# 01 道路清洗示例

本 Notebook 将 **ArcGIS Pro 中经过 Integrate（0.5 m）与 Dissolve 预处理的道路数据**作为输入，执行分网格道路规整、节点整合、建筑约束延伸、安全缺口修复、阶段审计与结果回退。

## 运行前目录

```text
项目根目录/
├── 02_精简版代码/
│   └── 01_道路清洗示例.ipynb
├── 03_小范围样例数据/
│   ├── boundary_sample.parquet
│   ├── streets_before_sample.parquet
│   └── buildings_sample.parquet
└── 04_示例输出/
```

默认使用最多 2 个进程，适合小范围演示。完整项目曾使用 2000 m 网格、800 m 缓冲和 10 核并行；示例保留相同空间参数，但降低并行数。

In [ ]:
# 运行环境检查：仅检查，不自动安装
import importlib.util

REQUIRED_MODULES = ['numpy', 'pandas', 'geopandas', 'shapely', 'pyarrow', 'matplotlib', 'neatnet', 'joblib', 'tqdm']

missing_modules = [
    module
    for module in REQUIRED_MODULES
    if importlib.util.find_spec(module) is None
]

if missing_modules:
    raise ImportError(
        "当前环境缺少以下依赖："
        + ", ".join(missing_modules)
        + "\n可在终端或新的 Notebook Cell 中安装："
        + "\npython -m pip install numpy pandas geopandas shapely pyarrow matplotlib neatnet joblib tqdm psutil pyogrio"
    )

print("依赖检查通过。")


## 1. 环境、路径与参数

核心参数包括：NeatNet 规整 10 m、节点整合 8 m、建筑约束延伸 15 m、安全缺口容差 5 m。参数仅用于展示本项目的处理逻辑，迁移到其他城市和数据源时应重新校准。

In [ ]:
# ============================================================
# 道路清洗示例 V13
# 建筑障碍约束 + 阶段审计 + 安全缺口修复
#
# 流程：
# ArcGIS预处理路网
# → neatify（阶段审计）
# → consolidate_nodes（阶段审计）
# → extend_lines（直接使用原生建筑barrier）
# → 自定义close_gaps（逐组判断建筑穿越）
# → 网格拼接、全局节点化与输出
#
# 主要特点：
# 1. extend_lines完全沿用案例D的有障碍分支
# 2. 不再允许extend_lines失败后切换到无障碍模式
# 3. close_gaps不再整阶段回退
# 4. 安全缺口正常保留，穿越建筑的缺口单独拒绝
# 5. 保存逐阶段审计、接受/拒绝的缺口连接及运行性能
# ============================================================

import datetime
import os
import shutil
import threading
import time
import traceback
import warnings

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import shapely.ops as so

from shapely.geometry import (
    Point,
    LineString,
    box,
)

from joblib import Parallel, delayed
from tqdm.auto import tqdm

import neatnet
import matplotlib.pyplot as plt
import matplotlib as mpl

from IPython.display import display


try:
    import psutil

    PSUTIL_AVAILABLE = True

except ImportError:
    PSUTIL_AVAILABLE = False


# ============================================================
# 0. 警告与中文字体
# ============================================================

# NumPy longdouble警告通常不影响GeoPandas/Shapely的float64计算
warnings.filterwarnings(
    "ignore",
    message=(
        r"Signature .*numpy\.longdouble.*"
        r"does not match any known type.*"
    ),
    category=UserWarning,
)


mpl.rcParams["font.family"] = "sans-serif"

mpl.rcParams["font.sans-serif"] = [
    "WenQuanYi Micro Hei",
    "WenQuanYi Zen Hei",
    "Microsoft YaHei",
    "SimHei",
    "DejaVu Sans",
]

mpl.rcParams["axes.unicode_minus"] = False


# ============================================================
# 1. 环境与路径
# ============================================================

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

TARGET_CRS = "EPSG:4547"
REGION_ID = 0


def locate_project_dir():
    """定位包含样例数据目录的项目根目录。"""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for candidate in candidates:
        if (candidate / "03_小范围样例数据").exists():
            return candidate
    if cwd.name == "02_精简版代码":
        return cwd.parent
    return cwd

PROJECT_DIR = locate_project_dir()
DATA_DIR = PROJECT_DIR / "03_小范围样例数据"
OUTPUT_DIR = PROJECT_DIR / "04_示例输出"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 注意：streets_before_sample.parquet 是 ArcGIS Integrate(0.5 m)
# 与 Dissolve 后的道路输入，不是最初的 Vector Tile 原始数据。
REGIONS_HULL_PATH = DATA_DIR / "boundary_sample.parquet"
RAW_STREETS_PATH = DATA_DIR / "streets_before_sample.parquet"
BUILDINGS_PATH = DATA_DIR / "buildings_sample.parquet"


# ============================================================
# 2. 网格与并行参数
# ============================================================

GRID_SIZE = 2000
BUFFER_SIZE = 800
WORKER_CORES = min(2, os.cpu_count() or 1)

PRE_DISPATCH = "1.5*n_jobs"


# ============================================================
# 3. 道路清洗参数
# ============================================================

# neatify
NEATIFY_CONSOLIDATION_TOLERANCE_M = 10.0
NEATIFY_SIMPLIFICATION_FACTOR = 0.35
NEATIFY_MIN_DANGLE_LENGTH_M = 20.0
NEATIFY_N_LOOPS = 2
NEATIFY_ANGLE_THRESHOLD_DEG = 45.0

# consolidate_nodes
CONSOLIDATE_TOLERANCE_M = 8.0
CONSOLIDATE_PRESERVE_ENDS = True

# extend_lines：直接使用建筑barrier
EXTEND_LINES_TOLERANCE_M = 15.0

# 自定义close_gaps
CLOSE_GAPS_TOLERANCE_M = 5.0


# ============================================================
# 4. 建筑约束与审计参数
# ============================================================

# 质量审计使用内缩建筑面，避免仅贴建筑边界被计为穿越
BUILDING_AUDIT_INSET_M = 0.30

# close_gaps安全检查使用完整建筑面
# 只有在建筑内部形成线性相交时才拒绝
MAX_CLOSE_GAP_BUILDING_CROSS_M = 0.01

# 阶段前后道路匹配容差
CROSSING_MATCH_TOLERANCE_M = 0.50

# neatify和consolidate_nodes允许的新增建筑穿越
MAX_STAGE_NEW_BUILDING_CROSSING_M = 0.20

# extend_lines启用barrier后允许的数值误差
MAX_EXTEND_NEW_BUILDING_CROSSING_M = 0.20

# close_gaps结束后的新增穿越安全阈值
MAX_CLOSE_STAGE_NEW_CROSSING_M = 0.02

# 坐标标准化
COORDINATE_DECIMALS = 3

# 输入和输出道路最短长度
MIN_LINE_LENGTH_M = 0.50

# 缺口连接最短记录长度
MIN_GAP_CONNECTOR_LENGTH_M = 0.01

# 同一条道路的首尾端点不互相连接
EXCLUDE_SAME_LINE_GAP = True

# 是否要求每个网格都具有建筑面
REQUIRE_BUILDING_BARRIER = True


# ============================================================
# 5. 输出路径
# ============================================================

SAFE_BASENAME = "road_cleaning_sample"

OUTPUT_PARQUET = OUTPUT_DIR / "streets_cleaned_sample.parquet"
OUTPUT_GPKG = OUTPUT_DIR / f"{SAFE_BASENAME}.gpkg"
OUTPUT_PREVIEW = OUTPUT_DIR / "road_cleaning_preview.png"
OUTPUT_GRID_STATUS = OUTPUT_DIR / f"{SAFE_BASENAME}_grid_status.csv"
OUTPUT_STAGE_AUDIT = OUTPUT_DIR / f"{SAFE_BASENAME}_stage_audit.csv"
OUTPUT_CLOSE_GAP_GPKG = OUTPUT_DIR / f"{SAFE_BASENAME}_close_gap_candidates.gpkg"
OUTPUT_PERFORMANCE = OUTPUT_DIR / f"{SAFE_BASENAME}_performance.csv"
OUTPUT_FINAL_CROSSING = OUTPUT_DIR / f"{SAFE_BASENAME}_final_crossing_summary.csv"

# 示例输出写入独立目录，因此默认不备份旧文件。
BACKUP_EXISTING_OUTPUT = False


# ============================================================


## 2. 通用几何与内存工具

In [ ]:
# 6. 内存监控
# ============================================================

_memory_monitor_stop = threading.Event()
_peak_memory_mb = 0.0


def monitor_parent_and_children_memory():
    """
    记录Jupyter主进程和Joblib子进程的RSS总和峰值。
    """

    global _peak_memory_mb

    if not PSUTIL_AVAILABLE:
        return


    parent_process = psutil.Process(
        os.getpid()
    )


    while not _memory_monitor_stop.is_set():

        process_list = [
            parent_process
        ]


        try:

            process_list.extend(
                parent_process.children(
                    recursive=True
                )
            )

        except psutil.Error:

            pass


        total_rss_bytes = 0


        for current_process in process_list:

            try:

                total_rss_bytes += (
                    current_process
                    .memory_info()
                    .rss
                )

            except psutil.Error:

                continue


        current_memory_mb = (
            total_rss_bytes
            / 1024**2
        )


        _peak_memory_mb = max(
            _peak_memory_mb,
            current_memory_mb,
        )


        time.sleep(
            0.10
        )


# ============================================================
# 7. 通用几何工具
# ============================================================

def nonempty_geometry_mask(
    geometry_series,
):
    """
    同时排除None与EMPTY几何，
    避免GeoSeries.notna行为变化警告。
    """

    not_empty = (
        ~geometry_series.is_empty
    )


    result = pd.Series(
        False,
        index=geometry_series.index,
        dtype=bool,
    )


    candidate_index = (
        geometry_series.index[
            not_empty
        ]
    )


    if len(candidate_index) > 0:

        result.loc[
            candidate_index
        ] = (
            geometry_series
            .loc[candidate_index]
            .notna()
        )


    return result


def generate_grids(
    bounds,
    grid_size,
):
    """
    生成规则网格。
    """

    xmin, ymin, xmax, ymax = bounds

    geometries = []


    for x0 in np.arange(
        xmin,
        xmax,
        grid_size,
    ):

        for y0 in np.arange(
            ymin,
            ymax,
            grid_size,
        ):

            geometries.append(
                box(
                    x0,
                    y0,
                    x0 + grid_size,
                    y0 + grid_size,
                )
            )


    return gpd.GeoDataFrame(
        geometry=geometries,
        crs=TARGET_CRS,
    )


def iter_line_geometries(
    geometry,
):
    """
    从复杂几何中递归提取LineString。
    """

    if (
        geometry is None
        or geometry.is_empty
    ):
        return


    if geometry.geom_type == "LineString":

        if (
            geometry.length > 0
            and len(geometry.coords) >= 2
        ):

            yield geometry


    elif geometry.geom_type in [
        "MultiLineString",
        "GeometryCollection",
    ]:

        for part in geometry.geoms:

            yield from iter_line_geometries(
                part
            )


def iter_polygon_geometries(
    geometry,
):
    """
    从复杂几何中递归提取Polygon。
    """

    if (
        geometry is None
        or geometry.is_empty
    ):
        return


    if geometry.geom_type == "Polygon":

        yield geometry


    elif geometry.geom_type in [
        "MultiPolygon",
        "GeometryCollection",
    ]:

        for part in geometry.geoms:

            yield from iter_polygon_geometries(
                part
            )


def ensure_geodataframe(
    data,
    crs=TARGET_CRS,
):
    """
    将GeoSeries或GeoDataFrame统一为GeoDataFrame。
    """

    if isinstance(
        data,
        gpd.GeoDataFrame,
    ):

        result = data.copy()


    elif isinstance(
        data,
        gpd.GeoSeries,
    ):

        result = gpd.GeoDataFrame(
            geometry=data,
            crs=data.crs or crs,
        )


    else:

        result = gpd.GeoDataFrame(
            geometry=list(data),
            crs=crs,
        )


    if result.crs is None:

        result = result.set_crs(
            crs,
            allow_override=True,
        )


    elif result.crs.to_string() != crs:

        result = result.to_crs(
            crs
        )


    return result


def geometry_to_line_gdf(
    geometry,
    minimum_length_m=0.0,
    extra_attributes=None,
):
    """
    将复杂线几何拆为GeoDataFrame。
    """

    records = []

    extra_attributes = (
        extra_attributes or {}
    )


    for part_id, line in enumerate(
        iter_line_geometries(
            geometry
        )
    ):

        line_length = float(
            line.length
        )


        if line_length <= minimum_length_m:
            continue


        record = {
            "part_id": part_id,
            "length_m": line_length,
            "geometry": line,
        }


        record.update(
            extra_attributes
        )


        records.append(
            record
        )


    if not records:

        return gpd.GeoDataFrame(
            {
                "geometry": []
            },
            geometry="geometry",
            crs=TARGET_CRS,
        )


    return gpd.GeoDataFrame(
        records,
        geometry="geometry",
        crs=TARGET_CRS,
    )


def reconstruct_to_pure_linestrings(
    input_gdf,
    minimum_length_m=MIN_LINE_LENGTH_M,
):
    """
    转换为纯LineString并删除过短线。
    """

    input_gdf = ensure_geodataframe(
        input_gdf,
        TARGET_CRS,
    )


    pure_lines = []


    for geometry in input_gdf.geometry:

        if (
            geometry is None
            or geometry.is_empty
        ):
            continue


        pure_lines.extend(
            iter_line_geometries(
                geometry
            )
        )


    result = gpd.GeoDataFrame(
        geometry=pure_lines,
        crs=TARGET_CRS,
    )


    if result.empty:
        return result


    valid_mask = (
        nonempty_geometry_mask(
            result.geometry
        )
        & result.geometry.is_valid
        & (
            result.geom_type
            == "LineString"
        )
        & (
            result.geometry.length
            > minimum_length_m
        )
    )


    return (
        result.loc[
            valid_mask
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


def normalize_and_deduplicate_lines(
    line_gdf,
):
    """
    标准化线方向，删除正反方向完全重复线。
    """

    result = reconstruct_to_pure_linestrings(
        line_gdf,
        minimum_length_m=0.0,
    )


    if result.empty:
        return result


    try:

        normalized = shapely.normalize(
            result.geometry.array
        )

    except Exception:

        normalized = result.geometry.array


    result["_wkb"] = shapely.to_wkb(
        normalized,
        hex=True,
    )


    result = (
        result
        .drop_duplicates(
            subset="_wkb"
        )
        .drop(
            columns="_wkb"
        )
        .reset_index(
            drop=True
        )
    )


    return result


def unique_line_union(
    line_gdf,
):
    """
    合并道路并消除完全重叠部分。
    """

    lines = reconstruct_to_pure_linestrings(
        line_gdf,
        minimum_length_m=0.0,
    )


    if lines.empty:

        return shapely.GeometryCollection()


    return shapely.union_all(
        lines.geometry.array
    )


def safe_geometry_length(
    geometry,
):
    """
    安全读取几何长度。
    """

    if (
        geometry is None
        or geometry.is_empty
    ):

        return 0.0


    return float(
        geometry.length
    )


def line_merge_safe(
    geometry,
):
    """
    兼容不同Shapely版本的LineMerge。
    """

    try:

        return shapely.line_merge(
            geometry
        )

    except Exception:

        return so.linemerge(
            geometry
        )


def clean_linework_before_neatify(
    streets,
):
    """
    neatify前：
    1. 提取纯线；
    2. union_all节点化与溶解重叠线；
    3. 标准化与去重。
    """

    streets = reconstruct_to_pure_linestrings(
        streets,
        minimum_length_m=MIN_LINE_LENGTH_M,
    )


    if streets.empty:
        return streets


    line_union = shapely.union_all(
        streets.geometry.array
    )


    streets = geometry_to_line_gdf(
        line_union,
        minimum_length_m=MIN_LINE_LENGTH_M,
    )


    streets = streets.drop(
        columns=[
            "part_id",
            "length_m",
        ],
        errors="ignore",
    )


    return normalize_and_deduplicate_lines(
        streets
    )


# ============================================================


## 3. 建筑障碍与穿越审计

In [ ]:
# 8. 建筑障碍数据
# ============================================================

def prepare_building_barriers(
    local_buildings,
    clip_geometry,
):
    """
    返回：
    1. extend_lines使用的建筑barrier GeoSeries；
    2. 完整建筑合并面；
    3. 向内收缩后的建筑审计面；
    4. 有效建筑数量。
    """

    if (
        local_buildings is None
        or local_buildings.empty
    ):

        return (
            None,
            shapely.GeometryCollection(),
            shapely.GeometryCollection(),
            0,
        )


    clipped = gpd.clip(
        local_buildings,
        clip_geometry,
    )


    if clipped.empty:

        return (
            None,
            shapely.GeometryCollection(),
            shapely.GeometryCollection(),
            0,
        )


    polygons = []


    for geometry in clipped.geometry:

        if (
            geometry is None
            or geometry.is_empty
        ):
            continue


        try:

            fixed = shapely.make_valid(
                geometry
            )

        except Exception:

            try:

                fixed = geometry.buffer(
                    0
                )

            except Exception:

                continue


        polygons.extend(
            iter_polygon_geometries(
                fixed
            )
        )


    if not polygons:

        return (
            None,
            shapely.GeometryCollection(),
            shapely.GeometryCollection(),
            0,
        )


    building_union = shapely.union_all(
        polygons
    )


    building_barrier = gpd.GeoSeries(
        [
            building_union
        ],
        crs=TARGET_CRS,
        name="geometry",
    )


    inner_series = gpd.GeoSeries(
        polygons,
        crs=TARGET_CRS,
    ).buffer(
        -BUILDING_AUDIT_INSET_M
    )


    inner_mask = nonempty_geometry_mask(
        inner_series
    )


    inner_series = inner_series.loc[
        inner_mask
    ]


    if inner_series.empty:

        building_inner_union = (
            shapely.GeometryCollection()
        )


    else:

        building_inner_union = (
            shapely.union_all(
                inner_series.array
            )
        )


    return (
        building_barrier,
        building_union,
        building_inner_union,
        len(polygons),
    )


# ============================================================
# 9. 建筑穿越变化审计
# ============================================================

def calculate_crossing_change(
    before_gdf,
    after_gdf,
    building_inner_union,
):
    """
    计算阶段前后建筑内部道路变化。
    """

    before_union = unique_line_union(
        before_gdf
    )

    after_union = unique_line_union(
        after_gdf
    )


    if (
        building_inner_union is None
        or building_inner_union.is_empty
    ):

        return {
            "before_road_length_m": (
                safe_geometry_length(
                    before_union
                )
            ),
            "after_road_length_m": (
                safe_geometry_length(
                    after_union
                )
            ),
            "before_crossing_m": 0.0,
            "after_crossing_m": 0.0,
            "introduced_crossing_m": 0.0,
            "eliminated_crossing_m": 0.0,
            "net_crossing_change_m": 0.0,
            "introduced_geometry": (
                shapely.GeometryCollection()
            ),
        }


    before_crossing = before_union.intersection(
        building_inner_union
    )

    after_crossing = after_union.intersection(
        building_inner_union
    )


    if before_crossing.is_empty:

        before_match_area = (
            shapely.GeometryCollection()
        )

    else:

        before_match_area = before_crossing.buffer(
            CROSSING_MATCH_TOLERANCE_M,
            cap_style=2,
            join_style=2,
        )


    if after_crossing.is_empty:

        after_match_area = (
            shapely.GeometryCollection()
        )

    else:

        after_match_area = after_crossing.buffer(
            CROSSING_MATCH_TOLERANCE_M,
            cap_style=2,
            join_style=2,
        )


    introduced = after_crossing.difference(
        before_match_area
    )

    eliminated = before_crossing.difference(
        after_match_area
    )


    before_cross_length = safe_geometry_length(
        before_crossing
    )

    after_cross_length = safe_geometry_length(
        after_crossing
    )


    return {
        "before_road_length_m": (
            safe_geometry_length(
                before_union
            )
        ),
        "after_road_length_m": (
            safe_geometry_length(
                after_union
            )
        ),
        "before_crossing_m": (
            before_cross_length
        ),
        "after_crossing_m": (
            after_cross_length
        ),
        "introduced_crossing_m": (
            safe_geometry_length(
                introduced
            )
        ),
        "eliminated_crossing_m": (
            safe_geometry_length(
                eliminated
            )
        ),
        "net_crossing_change_m": (
            after_cross_length
            - before_cross_length
        ),
        "introduced_geometry": introduced,
    }


# ============================================================


## 4. 分阶段保护、道路延伸与安全缺口修复

In [ ]:
# 10. neatify / consolidate_nodes整阶段防护
# ============================================================

def apply_whole_stage_guard(
    grid_id,
    stage_name,
    before_gdf,
    stage_function,
    building_inner_union,
):
    """
    对会整体移动或重构道路的阶段进行审计。

    新增建筑穿越超过阈值时，
    只回退当前阶段。
    """

    stage_start = time.perf_counter()


    candidate = stage_function(
        before_gdf.copy()
    )


    candidate = ensure_geodataframe(
        candidate,
        TARGET_CRS,
    )


    candidate = reconstruct_to_pure_linestrings(
        candidate,
        minimum_length_m=MIN_LINE_LENGTH_M,
    )


    audit = calculate_crossing_change(
        before_gdf,
        candidate,
        building_inner_union,
    )


    accepted = (
        audit["introduced_crossing_m"]
        <= MAX_STAGE_NEW_BUILDING_CROSSING_M
    )


    if accepted:

        result = candidate
        message = "阶段通过"

    else:

        result = before_gdf.copy()
        message = "新增建筑穿越超限，当前阶段回退"


    rejected_gdf = geometry_to_line_gdf(
        audit["introduced_geometry"],
        minimum_length_m=0.01,
        extra_attributes={
            "grid_id": grid_id,
            "stage": stage_name,
            "status": "rejected",
            "reason": (
                "whole_stage_building_crossing"
            ),
        },
    )


    record = {
        "grid_id": grid_id,
        "stage": stage_name,
        "guard_mode": "whole_stage",
        "accepted": accepted,
        "rollback": not accepted,
        "before_feature_count": len(
            before_gdf
        ),
        "candidate_feature_count": len(
            candidate
        ),
        "result_feature_count": len(
            result
        ),
        "before_road_length_m": (
            audit["before_road_length_m"]
        ),
        "candidate_road_length_m": (
            audit["after_road_length_m"]
        ),
        "before_crossing_m": (
            audit["before_crossing_m"]
        ),
        "candidate_crossing_m": (
            audit["after_crossing_m"]
        ),
        "introduced_crossing_m": (
            audit["introduced_crossing_m"]
        ),
        "eliminated_crossing_m": (
            audit["eliminated_crossing_m"]
        ),
        "accepted_gap_count": np.nan,
        "rejected_gap_count": np.nan,
        "stage_seconds": (
            time.perf_counter()
            - stage_start
        ),
        "message": message,
    }


    return (
        result,
        record,
        rejected_gdf,
    )


# ============================================================
# 11. extend_lines：直接沿用案例D原生barrier
# ============================================================

def apply_extend_lines_with_native_barrier(
    grid_id,
    before_gdf,
    building_barrier,
    building_inner_union,
):
    """
    直接运行案例D中的有建筑障碍分支。

    不存在无障碍回退。
    barrier执行失败时直接报错。
    """

    stage_start = time.perf_counter()


    if building_barrier is None:

        raise RuntimeError(
            "extend_lines缺少建筑barrier。"
        )


    try:

        candidate = neatnet.extend_lines(
            gdf=before_gdf.copy(),
            tolerance=(
                EXTEND_LINES_TOLERANCE_M
            ),
            barrier=building_barrier,
        )

    except Exception as error:

        raise RuntimeError(
            f"Grid {grid_id} 的extend_lines"
            f"建筑barrier执行失败：{error}"
        ) from error


    candidate = ensure_geodataframe(
        candidate,
        TARGET_CRS,
    )


    candidate = reconstruct_to_pure_linestrings(
        candidate,
        minimum_length_m=MIN_LINE_LENGTH_M,
    )


    # extend_lines可能连接到另一条线的中部，
    # 在进入close_gaps前先进行平面节点化
    candidate_union = shapely.union_all(
        candidate.geometry.array
    )


    candidate = geometry_to_line_gdf(
        candidate_union,
        minimum_length_m=MIN_LINE_LENGTH_M,
    )


    candidate = candidate.drop(
        columns=[
            "part_id",
            "length_m",
        ],
        errors="ignore",
    )


    candidate = normalize_and_deduplicate_lines(
        candidate
    )


    audit = calculate_crossing_change(
        before_gdf,
        candidate,
        building_inner_union,
    )


    accepted = (
        audit["introduced_crossing_m"]
        <= MAX_EXTEND_NEW_BUILDING_CROSSING_M
    )


    if accepted:

        result = candidate
        message = "原生建筑barrier延伸通过"

    else:

        result = before_gdf.copy()

        message = (
            "启用barrier后仍检测到新增建筑穿越，"
            "extend_lines阶段回退"
        )


    rejected_gdf = geometry_to_line_gdf(
        audit["introduced_geometry"],
        minimum_length_m=0.01,
        extra_attributes={
            "grid_id": grid_id,
            "stage": "extend_lines",
            "status": "rejected",
            "reason": (
                "native_barrier_residual_crossing"
            ),
        },
    )


    record = {
        "grid_id": grid_id,
        "stage": "extend_lines",
        "guard_mode": "native_building_barrier",
        "accepted": accepted,
        "rollback": not accepted,
        "before_feature_count": len(
            before_gdf
        ),
        "candidate_feature_count": len(
            candidate
        ),
        "result_feature_count": len(
            result
        ),
        "before_road_length_m": (
            audit["before_road_length_m"]
        ),
        "candidate_road_length_m": (
            audit["after_road_length_m"]
        ),
        "before_crossing_m": (
            audit["before_crossing_m"]
        ),
        "candidate_crossing_m": (
            audit["after_crossing_m"]
        ),
        "introduced_crossing_m": (
            audit["introduced_crossing_m"]
        ),
        "eliminated_crossing_m": (
            audit["eliminated_crossing_m"]
        ),
        "accepted_gap_count": np.nan,
        "rejected_gap_count": np.nan,
        "stage_seconds": (
            time.perf_counter()
            - stage_start
        ),
        "message": message,
    }


    return (
        result,
        record,
        rejected_gdf,
    )


# ============================================================
# 12. 建筑安全close_gaps
# ============================================================

def coordinate_key(
    coordinate,
):
    """
    将坐标转换为稳定的端点编号。
    """

    return (
        round(
            float(coordinate[0]),
            COORDINATE_DECIMALS,
        ),
        round(
            float(coordinate[1]),
            COORDINATE_DECIMALS,
        ),
    )


def prepare_endpoint_table(
    streets,
):
    """
    提取每条LineString的首尾端点。
    """

    records = []


    for line_id, geometry in enumerate(
        streets.geometry
    ):

        coordinates = list(
            geometry.coords
        )


        if len(coordinates) < 2:
            continue


        for endpoint_side, coordinate in [
            (
                "start",
                coordinates[0],
            ),
            (
                "end",
                coordinates[-1],
            ),
        ]:

            point = Point(
                coordinate
            )


            records.append({
                "endpoint_id": len(
                    records
                ),
                "line_id": line_id,
                "side": endpoint_side,
                "coord_key": coordinate_key(
                    coordinate
                ),
                "geometry": point,
            })


    endpoints = gpd.GeoDataFrame(
        records,
        geometry="geometry",
        crs=TARGET_CRS,
    )


    if endpoints.empty:
        return endpoints


    coordinate_counts = (
        endpoints[
            "coord_key"
        ].value_counts()
    )


    endpoints[
        "endpoint_count"
    ] = endpoints[
        "coord_key"
    ].map(
        coordinate_counts
    )


    # 在端点表示下，出现一次的坐标视为悬挂端点
    endpoints[
        "is_dangle"
    ] = (
        endpoints[
            "endpoint_count"
        ]
        == 1
    )


    return endpoints


def find_nearest_endpoint_targets(
    endpoints,
    tolerance_m,
):
    """
    为每个度为1的端点寻找容差内最近目标。

    目标可以是：
    - 另一个悬挂端点；
    - 已有道路交会端点。

    悬挂端点之间只接受互为最近邻的配对。
    """

    nearest_target = {}


    if endpoints.empty:
        return nearest_target


    spatial_index = endpoints.sindex


    dangles = endpoints[
        endpoints["is_dangle"]
    ]


    for source_index, source in dangles.iterrows():

        search_area = source.geometry.buffer(
            tolerance_m
        )


        candidate_positions = spatial_index.query(
            search_area,
            predicate="intersects",
        )


        if len(candidate_positions) == 0:
            continue


        candidates = endpoints.iloc[
            np.unique(
                candidate_positions
            )
        ].copy()


        # 排除相同坐标
        candidates = candidates[
            candidates["coord_key"]
            != source["coord_key"]
        ]


        if EXCLUDE_SAME_LINE_GAP:

            candidates = candidates[
                candidates["line_id"]
                != source["line_id"]
            ]


        if candidates.empty:
            continue


        candidates[
            "_distance"
        ] = candidates.geometry.distance(
            source.geometry
        )


        candidates = candidates[
            candidates["_distance"]
            <= tolerance_m
        ]


        if candidates.empty:
            continue


        # 先按距离排序；
        # 距离相同时优先已有连接节点
        candidates[
            "_connected_priority"
        ] = (
            -candidates[
                "endpoint_count"
            ]
        )


        candidates = candidates.sort_values(
            [
                "_distance",
                "_connected_priority",
                "line_id",
            ]
        )


        nearest_target[
            int(source_index)
        ] = int(
            candidates.index[0]
        )


    return nearest_target


def build_gap_proposals(
    endpoints,
    nearest_target,
):
    """
    将最近端点关系转换为缺口修复提案。

    情况A：
        悬挂端点 → 已连接端点
        仅移动悬挂端点至既有节点。

    情况B：
        悬挂端点 ↔ 悬挂端点
        只有互为最近邻时才吸附至中点。
    """

    proposals = []

    processed_pairs = set()


    for source_index, target_index in (
        nearest_target.items()
    ):

        source = endpoints.loc[
            source_index
        ]

        target = endpoints.loc[
            target_index
        ]


        # 目标是另一个悬挂端点
        if bool(
            target["is_dangle"]
        ):

            if (
                nearest_target.get(
                    target_index
                )
                != source_index
            ):
                continue


            pair_key = tuple(
                sorted([
                    source_index,
                    target_index,
                ])
            )


            if pair_key in processed_pairs:
                continue


            processed_pairs.add(
                pair_key
            )


            snap_x = (
                source.geometry.x
                + target.geometry.x
            ) / 2.0

            snap_y = (
                source.geometry.y
                + target.geometry.y
            ) / 2.0


            snap_point = Point(
                snap_x,
                snap_y,
            )


            members = [
                source_index,
                target_index,
            ]


            proposal_type = (
                "dangle_to_dangle"
            )


        # 目标是已有连接节点
        else:

            snap_point = target.geometry

            members = [
                source_index
            ]

            pair_key = (
                source_index,
                target["coord_key"],
            )


            if pair_key in processed_pairs:
                continue


            processed_pairs.add(
                pair_key
            )


            proposal_type = (
                "dangle_to_existing_node"
            )


        connector_lines = []


        for member_index in members:

            member_point = endpoints.loc[
                member_index,
                "geometry",
            ]


            connector = LineString([
                (
                    member_point.x,
                    member_point.y,
                ),
                (
                    snap_point.x,
                    snap_point.y,
                ),
            ])


            if (
                connector.length
                > MIN_GAP_CONNECTOR_LENGTH_M
            ):

                connector_lines.append(
                    connector
                )


        if not connector_lines:
            continue


        connector_union = shapely.union_all(
            connector_lines
        )


        connector_union = line_merge_safe(
            connector_union
        )


        gap_distance = max(
            endpoints.loc[
                member_index,
                "geometry",
            ].distance(
                snap_point
            )
            for member_index in members
        )


        proposals.append({
            "proposal_id": len(
                proposals
            ),
            "proposal_type": (
                proposal_type
            ),
            "source_endpoint_id": (
                int(source_index)
            ),
            "target_endpoint_id": (
                int(target_index)
            ),
            "member_endpoint_ids": (
                tuple(
                    int(value)
                    for value in members
                )
            ),
            "snap_x": float(
                snap_point.x
            ),
            "snap_y": float(
                snap_point.y
            ),
            "gap_distance_m": float(
                gap_distance
            ),
            "connector_length_m": (
                safe_geometry_length(
                    connector_union
                )
            ),
            "geometry": connector_union,
        })


    return proposals


def apply_building_safe_close_gaps(
    grid_id,
    before_gdf,
    building_union,
    building_inner_union,
):
    """
    对缺口提案逐组执行建筑检测。

    每个提案作为一个整体：
    - 连接线不穿越建筑：接受；
    - 任一连接线穿越建筑：拒绝整个提案。

    不会因为一处错误取消整个网格的close_gaps。
    """

    stage_start = time.perf_counter()


    streets = reconstruct_to_pure_linestrings(
        before_gdf,
        minimum_length_m=MIN_LINE_LENGTH_M,
    )


    endpoints = prepare_endpoint_table(
        streets
    )


    nearest_target = (
        find_nearest_endpoint_targets(
            endpoints,
            CLOSE_GAPS_TOLERANCE_M,
        )
    )


    proposals = build_gap_proposals(
        endpoints,
        nearest_target,
    )


    endpoint_updates = {}

    accepted_records = []
    rejected_records = []


    for proposal in proposals:

        connector_geometry = proposal[
            "geometry"
        ]


        if (
            building_union is None
            or building_union.is_empty
        ):

            building_cross_length = 0.0


        else:

            building_cross_length = (
                safe_geometry_length(
                    connector_geometry.intersection(
                        building_union
                    )
                )
            )


        accepted = (
            building_cross_length
            <= MAX_CLOSE_GAP_BUILDING_CROSS_M
        )


        record = {
            "grid_id": grid_id,
            "stage": "close_gaps",
            "proposal_id": (
                proposal[
                    "proposal_id"
                ]
            ),
            "proposal_type": (
                proposal[
                    "proposal_type"
                ]
            ),
            "gap_distance_m": (
                proposal[
                    "gap_distance_m"
                ]
            ),
            "connector_length_m": (
                proposal[
                    "connector_length_m"
                ]
            ),
            "building_crossing_m": (
                building_cross_length
            ),
            "accepted": accepted,
            "reason": (
                ""
                if accepted
                else "crosses_building"
            ),
            "geometry": (
                connector_geometry
            ),
        }


        if accepted:

            accepted_records.append(
                record
            )


            snap_coordinate = (
                proposal["snap_x"],
                proposal["snap_y"],
            )


            for endpoint_id in (
                proposal[
                    "member_endpoint_ids"
                ]
            ):

                endpoint_row = endpoints.loc[
                    endpoint_id
                ]


                update_key = (
                    int(
                        endpoint_row[
                            "line_id"
                        ]
                    ),
                    endpoint_row[
                        "side"
                    ],
                )


                endpoint_updates[
                    update_key
                ] = snap_coordinate


        else:

            rejected_records.append(
                record
            )


    # --------------------------------------------------------
    # 根据接受的缺口提案修改道路端点
    # --------------------------------------------------------

    updated_geometries = []


    for line_id, geometry in enumerate(
        streets.geometry
    ):

        coordinates = list(
            geometry.coords
        )


        start_update = endpoint_updates.get(
            (
                line_id,
                "start",
            )
        )


        end_update = endpoint_updates.get(
            (
                line_id,
                "end",
            )
        )


        if start_update is not None:

            coordinates[0] = (
                start_update
            )


        if end_update is not None:

            coordinates[-1] = (
                end_update
            )


        updated_geometry = LineString(
            coordinates
        )


        if (
            updated_geometry.length
            > MIN_LINE_LENGTH_M
        ):

            updated_geometries.append(
                updated_geometry
            )


    result = gpd.GeoDataFrame(
        geometry=updated_geometries,
        crs=TARGET_CRS,
    )


    # 节点化并溶解可能产生的微小重复
    if not result.empty:

        result_union = shapely.union_all(
            result.geometry.array
        )


        result = geometry_to_line_gdf(
            result_union,
            minimum_length_m=MIN_LINE_LENGTH_M,
        )


        result = result.drop(
            columns=[
                "part_id",
                "length_m",
            ],
            errors="ignore",
        )


        result = normalize_and_deduplicate_lines(
            result
        )


    audit = calculate_crossing_change(
        before_gdf,
        result,
        building_inner_union,
    )


    emergency_rollback = (
        audit["introduced_crossing_m"]
        > MAX_CLOSE_STAGE_NEW_CROSSING_M
    )


    if emergency_rollback:

        result = before_gdf.copy()

        message = (
            "选择性close_gaps后仍产生建筑穿越，"
            "触发紧急阶段回退"
        )


    else:

        message = (
            f"接受{len(accepted_records)}组安全缺口，"
            f"拒绝{len(rejected_records)}组建筑冲突缺口"
        )


    if accepted_records:

        accepted_gdf = gpd.GeoDataFrame(
            accepted_records,
            geometry="geometry",
            crs=TARGET_CRS,
        )

    else:

        accepted_gdf = gpd.GeoDataFrame(
            {
                "geometry": []
            },
            geometry="geometry",
            crs=TARGET_CRS,
        )


    if rejected_records:

        rejected_gdf = gpd.GeoDataFrame(
            rejected_records,
            geometry="geometry",
            crs=TARGET_CRS,
        )

    else:

        rejected_gdf = gpd.GeoDataFrame(
            {
                "geometry": []
            },
            geometry="geometry",
            crs=TARGET_CRS,
        )


    record = {
        "grid_id": grid_id,
        "stage": "close_gaps",
        "guard_mode": (
            "proposal_level_building_barrier"
        ),
        "accepted": (
            not emergency_rollback
        ),
        "rollback": emergency_rollback,
        "before_feature_count": len(
            before_gdf
        ),
        "candidate_feature_count": len(
            result
        ),
        "result_feature_count": len(
            result
        ),
        "before_road_length_m": (
            audit["before_road_length_m"]
        ),
        "candidate_road_length_m": (
            audit["after_road_length_m"]
        ),
        "before_crossing_m": (
            audit["before_crossing_m"]
        ),
        "candidate_crossing_m": (
            audit["after_crossing_m"]
        ),
        "introduced_crossing_m": (
            audit["introduced_crossing_m"]
            if not emergency_rollback
            else 0.0
        ),
        "eliminated_crossing_m": (
            audit["eliminated_crossing_m"]
        ),
        "accepted_gap_count": len(
            accepted_records
        ),
        "rejected_gap_count": len(
            rejected_records
        ),
        "accepted_gap_length_m": float(
            sum(
                row["connector_length_m"]
                for row in accepted_records
            )
        ),
        "rejected_gap_length_m": float(
            sum(
                row["connector_length_m"]
                for row in rejected_records
            )
        ),
        "stage_seconds": (
            time.perf_counter()
            - stage_start
        ),
        "message": message,
    }


    return (
        result,
        record,
        accepted_gdf,
        rejected_gdf,
    )


# ============================================================


## 5. 单网格处理

In [ ]:
# 13. 单网格处理
# ============================================================

def process_single_grid_chunk(
    grid_id,
    base_geom,
    local_streets,
    local_buildings,
):
    """
    执行单网格道路清洗。
    """

    grid_start = time.perf_counter()

    stage_records = []
    accepted_gap_frames = []
    rejected_change_frames = []


    try:

        halo_geom = base_geom.buffer(
            BUFFER_SIZE
        )


        streets = gpd.clip(
            local_streets,
            halo_geom,
        )


        if streets.empty:

            return {
                "grid_id": grid_id,
                "status": "empty_input",
                "message": "缓冲范围内无道路",
                "result": None,
                "stage_records": [],
                "accepted_gaps": [],
                "rejected_changes": [],
                "elapsed_seconds": (
                    time.perf_counter()
                    - grid_start
                ),
            }


        (
            building_barrier,
            building_union,
            building_inner_union,
            building_count,
        ) = prepare_building_barriers(
            local_buildings,
            halo_geom,
        )


        if (
            REQUIRE_BUILDING_BARRIER
            and building_barrier is None
        ):

            raise RuntimeError(
                "网格内没有有效建筑障碍。"
            )


        road_mask = (
            nonempty_geometry_mask(
                streets.geometry
            )
            & streets.geometry.is_valid
        )


        streets = (
            streets.loc[
                road_mask
            ]
            .copy()
        )


        if streets.empty:

            raise RuntimeError(
                "输入道路均为空或无效。"
            )


        streets["geometry"] = (
            shapely.force_2d(
                streets.geometry.array
            )
        )


        initial_union = shapely.union_all(
            streets.geometry.array
        )


        streets = geometry_to_line_gdf(
            initial_union,
            minimum_length_m=MIN_LINE_LENGTH_M,
        )


        streets = streets.drop(
            columns=[
                "part_id",
                "length_m",
            ],
            errors="ignore",
        )


        streets = clean_linework_before_neatify(
            streets
        )


        if streets.empty:

            raise RuntimeError(
                "进入neatify前无有效道路。"
            )


        stage_0 = streets.copy()


        # ====================================================
        # 阶段1：neatify
        # ====================================================

        def run_neatify(
            input_streets,
        ):

            return neatnet.neatify(
                streets=input_streets,
                exclusion_mask=building_barrier,
                consolidation_tolerance=(
                    NEATIFY_CONSOLIDATION_TOLERANCE_M
                ),
                simplification_factor=(
                    NEATIFY_SIMPLIFICATION_FACTOR
                ),
                min_dangle_length=(
                    NEATIFY_MIN_DANGLE_LENGTH_M
                ),
                n_loops=(
                    NEATIFY_N_LOOPS
                ),
                angle_threshold=(
                    NEATIFY_ANGLE_THRESHOLD_DEG
                ),
            )


        (
            stage_1,
            audit_1,
            rejected_1,
        ) = apply_whole_stage_guard(
            grid_id=grid_id,
            stage_name="neatify",
            before_gdf=stage_0,
            stage_function=run_neatify,
            building_inner_union=(
                building_inner_union
            ),
        )


        stage_records.append(
            audit_1
        )


        if not rejected_1.empty:

            rejected_change_frames.append(
                rejected_1
            )


        # ====================================================
        # 阶段2：consolidate_nodes
        # ====================================================

        def run_consolidate_nodes(
            input_streets,
        ):

            return neatnet.consolidate_nodes(
                input_streets,
                tolerance=(
                    CONSOLIDATE_TOLERANCE_M
                ),
                preserve_ends=(
                    CONSOLIDATE_PRESERVE_ENDS
                ),
            )


        (
            stage_2,
            audit_2,
            rejected_2,
        ) = apply_whole_stage_guard(
            grid_id=grid_id,
            stage_name="consolidate_nodes",
            before_gdf=stage_1,
            stage_function=(
                run_consolidate_nodes
            ),
            building_inner_union=(
                building_inner_union
            ),
        )


        stage_records.append(
            audit_2
        )


        if not rejected_2.empty:

            rejected_change_frames.append(
                rejected_2
            )


        # ====================================================
        # 阶段3：extend_lines
        # 直接采用案例D原生barrier
        # ====================================================

        (
            stage_3,
            audit_3,
            rejected_3,
        ) = apply_extend_lines_with_native_barrier(
            grid_id=grid_id,
            before_gdf=stage_2,
            building_barrier=(
                building_barrier
            ),
            building_inner_union=(
                building_inner_union
            ),
        )


        stage_records.append(
            audit_3
        )


        if not rejected_3.empty:

            rejected_change_frames.append(
                rejected_3
            )


        # ====================================================
        # 阶段4：建筑安全close_gaps
        # ====================================================

        (
            stage_4,
            audit_4,
            accepted_gaps,
            rejected_gaps,
        ) = apply_building_safe_close_gaps(
            grid_id=grid_id,
            before_gdf=stage_3,
            building_union=(
                building_union
            ),
            building_inner_union=(
                building_inner_union
            ),
        )


        stage_records.append(
            audit_4
        )


        if not accepted_gaps.empty:

            accepted_gap_frames.append(
                accepted_gaps
            )


        if not rejected_gaps.empty:

            rejected_change_frames.append(
                rejected_gaps
            )


        # ====================================================
        # 输出归属到核心网格
        # ====================================================

        streets = reconstruct_to_pure_linestrings(
            stage_4,
            minimum_length_m=MIN_LINE_LENGTH_M,
        )


        if streets.empty:

            return {
                "grid_id": grid_id,
                "status": "empty_output",
                "message": "清洗后无有效道路",
                "result": None,
                "stage_records": stage_records,
                "accepted_gaps": (
                    accepted_gap_frames
                ),
                "rejected_changes": (
                    rejected_change_frames
                ),
                "elapsed_seconds": (
                    time.perf_counter()
                    - grid_start
                ),
            }


        representative_points = (
            streets.geometry.interpolate(
                0.5,
                normalized=True,
            )
        )


        core_mask = (
            representative_points.within(
                base_geom
            )
            | representative_points.touches(
                base_geom
            )
        )


        final_streets = (
            streets.loc[
                core_mask
            ]
            .copy()
        )


        final_streets = (
            normalize_and_deduplicate_lines(
                final_streets
            )
        )


        if final_streets.empty:

            return {
                "grid_id": grid_id,
                "status": "empty_output",
                "message": "核心网格内无输出道路",
                "result": None,
                "stage_records": stage_records,
                "accepted_gaps": (
                    accepted_gap_frames
                ),
                "rejected_changes": (
                    rejected_change_frames
                ),
                "elapsed_seconds": (
                    time.perf_counter()
                    - grid_start
                ),
            }


        return {
            "grid_id": grid_id,
            "status": "success",
            "message": "处理成功",
            "result": final_streets,
            "stage_records": stage_records,
            "accepted_gaps": (
                accepted_gap_frames
            ),
            "rejected_changes": (
                rejected_change_frames
            ),
            "input_feature_count": len(
                stage_0
            ),
            "output_feature_count": len(
                final_streets
            ),
            "building_count": building_count,
            "elapsed_seconds": (
                time.perf_counter()
                - grid_start
            ),
        }


    except Exception as error:

        return {
            "grid_id": grid_id,
            "status": "error",
            "message": (
                f"{type(error).__name__}: "
                f"{error}"
            ),
            "traceback": traceback.format_exc(),
            "result": None,
            "stage_records": stage_records,
            "accepted_gaps": (
                accepted_gap_frames
            ),
            "rejected_changes": (
                rejected_change_frames
            ),
            "elapsed_seconds": (
                time.perf_counter()
                - grid_start
            ),
        }


# ============================================================


## 6. 全区任务组织、汇总与输出

In [ ]:
# 14. 主程序
# ============================================================

def process_regions():

    global _peak_memory_mb


    start_datetime = (
        datetime.datetime.now()
    )

    start_time = time.perf_counter()


    _peak_memory_mb = 0.0
    _memory_monitor_stop.clear()


    memory_thread = None


    if PSUTIL_AVAILABLE:

        memory_thread = threading.Thread(
            target=(
                monitor_parent_and_children_memory
            ),
            daemon=True,
        )


        memory_thread.start()


    try:

        print("=" * 78)

        print(
            f"[{datetime.datetime.now()}] "
            "1. 读取道路、建筑和区域边界……"
        )


        for required_path in [
            REGIONS_HULL_PATH,
            RAW_STREETS_PATH,
            BUILDINGS_PATH,
        ]:

            if not required_path.exists():

                raise FileNotFoundError(
                    f"输入文件不存在："
                    f"{required_path}"
                )


        hulls = gpd.read_parquet(
            REGIONS_HULL_PATH
        ).to_crs(
            TARGET_CRS
        )


        if REGION_ID in hulls.index:
            region_hull = hulls.geometry.loc[REGION_ID]
        elif len(hulls) == 1:
            region_hull = hulls.geometry.iloc[0]
            print("边界文件仅含一个要素，已自动使用该要素。")
        else:
            raise KeyError(
                f"边界文件中不存在 REGION_ID={REGION_ID}，"
                "且文件不止一个要素。"
            )


        computation_hull = region_hull.buffer(
            BUFFER_SIZE
        )


        all_streets = gpd.read_parquet(
            RAW_STREETS_PATH
        ).to_crs(
            TARGET_CRS
        )


        road_mask = (
            nonempty_geometry_mask(
                all_streets.geometry
            )
            & all_streets.geometry.is_valid
            & all_streets.intersects(
                computation_hull
            )
        )


        all_streets = (
            all_streets.loc[
                road_mask
            ]
            .copy()
        )


        all_buildings = gpd.read_parquet(
            BUILDINGS_PATH
        ).to_crs(
            TARGET_CRS
        )


        building_mask = (
            nonempty_geometry_mask(
                all_buildings.geometry
            )
            & all_buildings.geometry.is_valid
            & all_buildings.intersects(
                computation_hull
            )
        )


        all_buildings = (
            all_buildings.loc[
                building_mask
            ]
            .copy()
        )


        print(
            "道路输入数：",
            f"{len(all_streets):,}"
        )

        print(
            "建筑输入数：",
            f"{len(all_buildings):,}"
        )

        print(
            "评价范围面积：",
            f"{region_hull.area / 1_000_000:.3f} km²"
        )


        # ----------------------------------------------------
        # 网格
        # ----------------------------------------------------

        base_grids = generate_grids(
            region_hull.bounds,
            GRID_SIZE,
        )


        base_grids = (
            base_grids[
                base_grids.intersects(
                    region_hull
                )
            ]
            .copy()
            .reset_index(
                drop=True
            )
        )


        print(
            f"[{datetime.datetime.now()}] "
            f"2. 生成{len(base_grids)}个测试网格……"
        )


        streets_sindex = (
            all_streets.sindex
        )

        buildings_sindex = (
            all_buildings.sindex
        )


        tasks = []


        for grid_id, base_geom in tqdm(
            enumerate(
                base_grids.geometry
            ),
            total=len(base_grids),
            desc="预切片",
        ):

            halo_geom = base_geom.buffer(
                BUFFER_SIZE
            )


            possible_street_index = list(
                streets_sindex.intersection(
                    halo_geom.bounds
                )
            )


            local_streets = (
                all_streets.iloc[
                    possible_street_index
                ]
            )


            local_streets = (
                local_streets[
                    local_streets.intersects(
                        halo_geom
                    )
                ]
                .copy()
            )


            if local_streets.empty:
                continue


            possible_building_index = list(
                buildings_sindex.intersection(
                    halo_geom.bounds
                )
            )


            local_buildings = (
                all_buildings.iloc[
                    possible_building_index
                ]
            )


            local_buildings = (
                local_buildings[
                    local_buildings.intersects(
                        halo_geom
                    )
                ]
                .copy()
            )


            tasks.append(
                (
                    grid_id,
                    base_geom,
                    local_streets,
                    local_buildings,
                )
            )


        print(
            f"[{datetime.datetime.now()}] "
            f"3. 启动{WORKER_CORES}核并行处理；"
            f"实际任务数：{len(tasks)}……"
        )


        results = Parallel(
            n_jobs=WORKER_CORES,
            pre_dispatch=PRE_DISPATCH,
            backend="loky",
        )(
            delayed(
                process_single_grid_chunk
            )(
                grid_id,
                base_geom,
                local_streets,
                local_buildings,
            )

            for (
                grid_id,
                base_geom,
                local_streets,
                local_buildings,
            ) in tqdm(
                tasks,
                desc="清洗进度",
                unit="网格",
            )
        )


        # ----------------------------------------------------
        # 网格状态
        # ----------------------------------------------------

        grid_status_records = []


        for result in results:

            grid_status_records.append({
                "grid_id": result[
                    "grid_id"
                ],
                "status": result[
                    "status"
                ],
                "message": result.get(
                    "message",
                    ""
                ),
                "input_feature_count": (
                    result.get(
                        "input_feature_count",
                        np.nan,
                    )
                ),
                "output_feature_count": (
                    result.get(
                        "output_feature_count",
                        np.nan,
                    )
                ),
                "building_count": (
                    result.get(
                        "building_count",
                        np.nan,
                    )
                ),
                "elapsed_seconds": (
                    result.get(
                        "elapsed_seconds",
                        np.nan,
                    )
                ),
                "traceback": result.get(
                    "traceback",
                    "",
                ),
            })


        grid_status_report = pd.DataFrame(
            grid_status_records
        )


        grid_status_report.to_csv(
            OUTPUT_GRID_STATUS,
            index=False,
            encoding="utf-8-sig",
        )


        # ----------------------------------------------------
        # 阶段审计
        # ----------------------------------------------------

        stage_records = []


        for result in results:

            stage_records.extend(
                result.get(
                    "stage_records",
                    []
                )
            )


        stage_report = pd.DataFrame(
            stage_records
        )


        stage_report.to_csv(
            OUTPUT_STAGE_AUDIT,
            index=False,
            encoding="utf-8-sig",
        )


        # ----------------------------------------------------
        # 接受与拒绝的缺口连接
        # ----------------------------------------------------

        accepted_gap_frames = []
        rejected_frames = []


        for result in results:

            accepted_gap_frames.extend(
                result.get(
                    "accepted_gaps",
                    []
                )
            )


            rejected_frames.extend(
                result.get(
                    "rejected_changes",
                    []
                )
            )


        accepted_gap_frames = [
            frame
            for frame in accepted_gap_frames
            if (
                frame is not None
                and not frame.empty
            )
        ]


        rejected_frames = [
            frame
            for frame in rejected_frames
            if (
                frame is not None
                and not frame.empty
            )
        ]


        if OUTPUT_CLOSE_GAP_GPKG.exists():

            OUTPUT_CLOSE_GAP_GPKG.unlink()


        gpkg_written = False


        if accepted_gap_frames:

            accepted_gaps_all = (
                gpd.GeoDataFrame(
                    pd.concat(
                        accepted_gap_frames,
                        ignore_index=True,
                    ),
                    geometry="geometry",
                    crs=TARGET_CRS,
                )
            )


            accepted_gaps_all.to_file(
                OUTPUT_CLOSE_GAP_GPKG,
                layer="accepted_close_gaps",
                driver="GPKG",
            )


            gpkg_written = True


        if rejected_frames:

            rejected_all = (
                gpd.GeoDataFrame(
                    pd.concat(
                        rejected_frames,
                        ignore_index=True,
                    ),
                    geometry="geometry",
                    crs=TARGET_CRS,
                )
            )


            rejected_all.to_file(
                OUTPUT_CLOSE_GAP_GPKG,
                layer="rejected_changes",
                driver="GPKG",
                mode=(
                    "a"
                    if gpkg_written
                    else "w"
                ),
            )


        # ----------------------------------------------------
        # 拼合成功结果
        # ----------------------------------------------------

        successful_results = [
            result["result"]
            for result in results
            if (
                result["status"] == "success"
                and result["result"] is not None
                and not result["result"].empty
            )
        ]


        if not successful_results:

            error_messages = (
                grid_status_report[
                    grid_status_report[
                        "status"
                    ]
                    == "error"
                ][
                    [
                        "grid_id",
                        "message",
                    ]
                ]
            )


            raise RuntimeError(
                "没有成功生成道路结果。\n"
                f"{error_messages.to_string(index=False)}"
            )


        print(
            f"[{datetime.datetime.now()}] "
            "4. 拼合网格结果并进行全局节点化……"
        )


        global_streets = gpd.GeoDataFrame(
            pd.concat(
                successful_results,
                ignore_index=True,
            ),
            geometry="geometry",
            crs=TARGET_CRS,
        )


        global_streets = (
            reconstruct_to_pure_linestrings(
                global_streets,
                minimum_length_m=MIN_LINE_LENGTH_M,
            )
        )


        global_streets = (
            normalize_and_deduplicate_lines(
                global_streets
            )
        )


        global_union = shapely.union_all(
            global_streets.geometry.array
        )


        global_merged = line_merge_safe(
            global_union
        )


        global_streets = geometry_to_line_gdf(
            global_merged,
            minimum_length_m=MIN_LINE_LENGTH_M,
        )


        global_streets = global_streets.drop(
            columns=[
                "part_id",
                "length_m",
            ],
            errors="ignore",
        )


        # 最终只保留真实评价范围
        global_streets = gpd.clip(
            global_streets,
            region_hull,
        )


        global_streets = (
            reconstruct_to_pure_linestrings(
                global_streets,
                minimum_length_m=MIN_LINE_LENGTH_M,
            )
        )


        global_streets = (
            normalize_and_deduplicate_lines(
                global_streets
            )
        )


        if global_streets.empty:

            raise RuntimeError(
                "最终裁剪后道路结果为空。"
            )


        # ----------------------------------------------------
        # 最终建筑穿越审计
        # ----------------------------------------------------

        (
            final_barrier,
            final_building_union,
            final_building_inner_union,
            final_building_count,
        ) = prepare_building_barriers(
            all_buildings,
            region_hull,
        )


        raw_core_streets = gpd.clip(
            all_streets,
            region_hull,
        )


        raw_core_streets = (
            reconstruct_to_pure_linestrings(
                raw_core_streets,
                minimum_length_m=0.0,
            )
        )


        final_audit = calculate_crossing_change(
            raw_core_streets,
            global_streets,
            final_building_inner_union,
        )


        final_crossing_report = pd.DataFrame([
            {
                "raw_crossing_m": (
                    final_audit[
                        "before_crossing_m"
                    ]
                ),
                "processed_crossing_m": (
                    final_audit[
                        "after_crossing_m"
                    ]
                ),
                "introduced_crossing_m": (
                    final_audit[
                        "introduced_crossing_m"
                    ]
                ),
                "eliminated_crossing_m": (
                    final_audit[
                        "eliminated_crossing_m"
                    ]
                ),
                "net_change_m": (
                    final_audit[
                        "net_crossing_change_m"
                    ]
                ),
                "raw_road_length_m": (
                    final_audit[
                        "before_road_length_m"
                    ]
                ),
                "processed_road_length_m": (
                    final_audit[
                        "after_road_length_m"
                    ]
                ),
                "building_count": (
                    final_building_count
                ),
                "building_inset_m": (
                    BUILDING_AUDIT_INSET_M
                ),
                "crossing_match_tolerance_m": (
                    CROSSING_MATCH_TOLERANCE_M
                ),
            }
        ])


        final_crossing_report.to_csv(
            OUTPUT_FINAL_CROSSING,
            index=False,
            encoding="utf-8-sig",
        )


        # ----------------------------------------------------
        # 备份旧输出
        # ----------------------------------------------------

        if (
            OUTPUT_PARQUET.exists()
            and BACKUP_EXISTING_OUTPUT
        ):

            timestamp = (
                datetime.datetime.now()
                .strftime(
                    "%Y%m%d_%H%M%S"
                )
            )


            backup_path = (
                OUTPUT_DIR
                / (
                    "streets_cleaned_sample_before_v13_"
                    f"{timestamp}.parquet"
                )
            )


            shutil.copy2(
                OUTPUT_PARQUET,
                backup_path,
            )


            print(
                "旧streets_0已备份：",
                backup_path,
            )


        # ----------------------------------------------------
        # 导出最终路网
        # ----------------------------------------------------

        if OUTPUT_GPKG.exists():

            OUTPUT_GPKG.unlink()


        global_streets.to_file(
            OUTPUT_GPKG,
            layer="processed_streets",
            driver="GPKG",
        )


        global_streets.to_parquet(
            OUTPUT_PARQUET,
            index=False,
        )


        # ----------------------------------------------------
        # 预览图
        # ----------------------------------------------------

        fig, ax = plt.subplots(
            figsize=(
                15,
                15,
            )
        )


        raw_core_streets.plot(
            ax=ax,
            color="#D9D9D9",
            linewidth=0.45,
            alpha=0.72,
            label="ArcGIS预处理输入路网",
            zorder=1,
        )


        global_streets.plot(
            ax=ax,
            color="#2F6B8A",
            linewidth=1.05,
            label="V13清洗结果",
            zorder=2,
        )


        gpd.GeoSeries(
            [
                region_hull
            ],
            crs=TARGET_CRS,
        ).boundary.plot(
            ax=ax,
            color="#333333",
            linewidth=0.80,
            zorder=3,
        )


        ax.set_title(
            (
                "广安门内道路清洗 V13｜"
                "案例D原生建筑障碍 + 安全缺口闭合"
            ),
            fontsize=16,
            pad=12,
        )


        ax.legend(
            loc="lower left",
            frameon=True,
        )


        ax.set_axis_off()
        ax.set_aspect(
            "equal"
        )


        fig.savefig(
            OUTPUT_PREVIEW,
            dpi=250,
            bbox_inches="tight",
            facecolor="white",
        )


        plt.close(
            fig
        )


        # ----------------------------------------------------
        # 性能与统计
        # ----------------------------------------------------

        elapsed_seconds = (
            time.perf_counter()
            - start_time
        )


        status_counts = (
            grid_status_report[
                "status"
            ].value_counts()
        )


        submitted_count = len(
            results
        )


        success_count = int(
            status_counts.get(
                "success",
                0,
            )
        )


        error_count = int(
            status_counts.get(
                "error",
                0,
            )
        )


        accepted_gap_count = (
            int(
                stage_report[
                    "accepted_gap_count"
                ]
                .fillna(0)
                .sum()
            )
            if (
                not stage_report.empty
                and "accepted_gap_count"
                in stage_report.columns
            )
            else 0
        )


        rejected_gap_count = (
            int(
                stage_report[
                    "rejected_gap_count"
                ]
                .fillna(0)
                .sum()
            )
            if (
                not stage_report.empty
                and "rejected_gap_count"
                in stage_report.columns
            )
            else 0
        )


        accepted_gap_length = (
            float(
                stage_report[
                    "accepted_gap_length_m"
                ]
                .fillna(0)
                .sum()
            )
            if (
                not stage_report.empty
                and "accepted_gap_length_m"
                in stage_report.columns
            )
            else 0.0
        )


        rejected_gap_length = (
            float(
                stage_report[
                    "rejected_gap_length_m"
                ]
                .fillna(0)
                .sum()
            )
            if (
                not stage_report.empty
                and "rejected_gap_length_m"
                in stage_report.columns
            )
            else 0.0
        )


        performance_report = pd.DataFrame([
            {
                "region_id": REGION_ID,
                "grid_size_m": GRID_SIZE,
                "buffer_size_m": BUFFER_SIZE,
                "worker_cores": WORKER_CORES,
                "generated_grid_count": len(
                    base_grids
                ),
                "submitted_grid_count": (
                    submitted_count
                ),
                "successful_grid_count": (
                    success_count
                ),
                "error_grid_count": (
                    error_count
                ),
                "processing_success_rate": (
                    success_count
                    / submitted_count
                    if submitted_count > 0
                    else np.nan
                ),
                "accepted_gap_count": (
                    accepted_gap_count
                ),
                "rejected_gap_count": (
                    rejected_gap_count
                ),
                "accepted_gap_length_m": (
                    accepted_gap_length
                ),
                "rejected_gap_length_m": (
                    rejected_gap_length
                ),
                "run_time_seconds": (
                    elapsed_seconds
                ),
                "peak_memory_mb": (
                    _peak_memory_mb
                    if PSUTIL_AVAILABLE
                    else np.nan
                ),
                "output_feature_count": len(
                    global_streets
                ),
                "output_length_m": float(
                    global_streets
                    .geometry
                    .length
                    .sum()
                ),
                "start_time": (
                    start_datetime.isoformat()
                ),
                "end_time": (
                    datetime.datetime.now()
                    .isoformat()
                ),
            }
        ])


        performance_report.to_csv(
            OUTPUT_PERFORMANCE,
            index=False,
            encoding="utf-8-sig",
        )


        # ----------------------------------------------------
        # 控制台输出
        # ----------------------------------------------------

        print("\n" + "=" * 78)
        print("V13处理完成")


        print(
            "成功网格：",
            f"{success_count}/"
            f"{submitted_count}"
        )


        print(
            "错误网格：",
            error_count
        )


        print(
            "接受的安全缺口：",
            (
                f"{accepted_gap_count}组，"
                f"{accepted_gap_length:,.2f} m"
            )
        )


        print(
            "拒绝的建筑冲突缺口：",
            (
                f"{rejected_gap_count}组，"
                f"{rejected_gap_length:,.2f} m"
            )
        )


        print(
            "输入建筑穿越：",
            (
                f"{final_audit['before_crossing_m']:,.2f} m"
            )
        )


        print(
            "最终建筑穿越：",
            (
                f"{final_audit['after_crossing_m']:,.2f} m"
            )
        )


        print(
            "最终新增建筑穿越：",
            (
                f"{final_audit['introduced_crossing_m']:,.6f} m"
            )
        )


        print(
            "运行时间：",
            f"{elapsed_seconds:.2f} s"
        )


        if PSUTIL_AVAILABLE:

            print(
                "峰值内存：",
                f"{_peak_memory_mb:.2f} MB"
            )


        print("\n输出文件：")


        print(
            "最终Parquet：",
            OUTPUT_PARQUET
        )


        print(
            "最终GPKG：",
            OUTPUT_GPKG
        )


        print(
            "阶段审计：",
            OUTPUT_STAGE_AUDIT
        )


        print(
            "网格状态：",
            OUTPUT_GRID_STATUS
        )


        print(
            "接受/拒绝缺口图层：",
            OUTPUT_CLOSE_GAP_GPKG
        )


        print(
            "最终穿越汇总：",
            OUTPUT_FINAL_CROSSING
        )


        return {
            "streets": global_streets,
            "grid_status": (
                grid_status_report
            ),
            "stage_audit": stage_report,
            "performance": (
                performance_report
            ),
            "final_crossing": (
                final_crossing_report
            ),
        }


    finally:

        _memory_monitor_stop.set()


        if memory_thread is not None:

            memory_thread.join(
                timeout=2.0
            )


# ============================================================


## 7. 执行与结果检查

运行后将生成标准化道路、网格状态、阶段审计、缺口候选、建筑穿越汇总和性能记录。

In [ ]:
# 15. 执行
# ============================================================

pipeline_result = process_regions()


# ============================================================
# 16. 显示关键结果
# ============================================================

if pipeline_result is not None:

    print("\n网格状态：")


    display(
        pipeline_result[
            "grid_status"
        ][
            [
                "grid_id",
                "status",
                "message",
                "input_feature_count",
                "output_feature_count",
                "building_count",
                "elapsed_seconds",
            ]
        ].style.format({
            "elapsed_seconds": "{:,.2f}",
        })
    )


    print("\n逐阶段审计：")


    stage_columns = [
        "grid_id",
        "stage",
        "guard_mode",
        "accepted",
        "rollback",
        "before_feature_count",
        "candidate_feature_count",
        "result_feature_count",
        "introduced_crossing_m",
        "eliminated_crossing_m",
        "accepted_gap_count",
        "rejected_gap_count",
        "accepted_gap_length_m",
        "rejected_gap_length_m",
        "stage_seconds",
        "message",
    ]


    stage_columns = [
        column
        for column in stage_columns
        if column in pipeline_result[
            "stage_audit"
        ].columns
    ]


    display(
        pipeline_result[
            "stage_audit"
        ][
            stage_columns
        ].style.format({
            "introduced_crossing_m": (
                "{:,.6f}"
            ),
            "eliminated_crossing_m": (
                "{:,.3f}"
            ),
            "accepted_gap_length_m": (
                "{:,.3f}"
            ),
            "rejected_gap_length_m": (
                "{:,.3f}"
            ),
            "stage_seconds": (
                "{:,.2f}"
            ),
        })
    )


    print("\n最终建筑穿越：")


    display(
        pipeline_result[
            "final_crossing"
        ].style.format({
            "raw_crossing_m": (
                "{:,.2f}"
            ),
            "processed_crossing_m": (
                "{:,.2f}"
            ),
            "introduced_crossing_m": (
                "{:,.6f}"
            ),
            "eliminated_crossing_m": (
                "{:,.2f}"
            ),
            "net_change_m": (
                "{:+,.2f}"
            ),
            "raw_road_length_m": (
                "{:,.2f}"
            ),
            "processed_road_length_m": (
                "{:,.2f}"
            ),
        })
    )


    print("\n运行性能：")


    display(
        pipeline_result[
            "performance"
        ].style.format({
            "processing_success_rate": (
                "{:.2%}"
            ),
            "accepted_gap_length_m": (
                "{:,.2f}"
            ),
            "rejected_gap_length_m": (
                "{:,.2f}"
            ),
            "run_time_seconds": (
                "{:,.2f}"
            ),
            "peak_memory_mb": (
                "{:,.2f}"
            ),
            "output_length_m": (
                "{:,.2f}"
            ),
        })
    )
